# Plant Growth Module (PGM) - DFS2 Map Generator

This notebook generates spatially distributed DFS2 maps for ECO Lab Plant Growth Module parameters based on land use classification and species-specific values.

## Workflow Overview
1. **Configure Paths**: Define input files and output directory
2. **Validate**: Check all required files exist
3. **Load Land Use Data**: Read spatial DFS2 grid and classification mapping
4. **Process Templates**: Read parameter templates and generate DFS2 maps for each parameter where type=1
5. **Verify**: Summarize generated files and validate output geometry


In [ ]:
from pathlib import Path
import warnings

# Import all required libraries
import pandas as pd
import numpy as np
import mikeio
import os
from plant_growth_module.pgm_helper import (
    VAL_COLS,
    CLASS_COLS,
    APPLY_COLS,
    ID_COLS,
    VALUE_COLS,
    KEY_COLS,
    TEMPLATE_COLS,
    TYPE_COLS,
    STATE_VARIABLE_SCOPE,
    find_col,
    confirm_columns,
    generate_dfs2_map,
    split_lu_mapping_by_apply,
    validate_paths,
)

warnings.filterwarnings("ignore")

## 📋 Step 0: Configuration

**Edit the paths below to match your data location.**

This cell defines:
- Input DFS2 file with land use spatial data
- Land use classification template (CODE → CLASS mapping)
- Parameter templates (Constants and Initial Conditions)
- Output directory for generated DFS2 files
- Processing options (AUTO_CONFIRM for batch mode)


In [10]:
# ═══════════════════════════════════════════════════════════════════════════
# 📥 INPUT FILES (ABSOLUTE PATHS)
# ═══════════════════════════════════════════════════════════════════════════
# Edit these paths as needed - use absolute paths only

# Land use spatial data (DFS2 format)
LANDUSE_DFS2 = Path(
    r"C:\Users\jaan\Repos\phishes-pdp\plant-growth-module\sample_data\landuse1_clo.dfs2"
)

# Land use classification mapping (CSV with CODE and CLASS columns)
LU_TEMPLATE = Path(
    r"C:\Users\jaan\Repos\phishes-pdp\plant-growth-module\sample_data\LU_template.csv"
)

# Soil profile spatial data and classification (required for soilprofile-based initial conditions)
SOILPROFILE_DFS2 = Path(
    r"C:\Users\jaan\Repos\phishes-pdp\plant-growth-module\sample_data\SoilProfile_gridcodes.dfs2"
)
SP_TEMPLATE = Path(
    r"C:\Users\jaan\Repos\phishes-pdp\plant-growth-module\sample_data\SP_template.csv"
)

# Template files for generating maps (CSV files)
TEMPLATE_FILES = [
    Path(
        r"C:\Users\jaan\Repos\phishes-pdp\plant-growth-module\sample_data\Constants_template.csv"
    ),
    Path(
        r"C:\Users\jaan\Repos\phishes-pdp\plant-growth-module\sample_data\InitConditions_template.csv"
    ),
]

# ═══════════════════════════════════════════════════════════════════════════
# 📤 OUTPUT DIRECTORY (ABSOLUTE PATH)
# ═══════════════════════════════════════════════════════════════════════════
OUTPUT_DIR = Path(r"C:\Users\jaan\Repos\phishes-pdp\plant-growth-module\output_data")

AUTO_CONFIRM = True


# Validate core paths
errors = validate_paths(LANDUSE_DFS2, LU_TEMPLATE, TEMPLATE_FILES, OUTPUT_DIR)

# Validate soil-profile support files
if not SOILPROFILE_DFS2.exists():
    errors.append(f"❌ Soil profile DFS2 not found: {SOILPROFILE_DFS2}")
if not SP_TEMPLATE.exists():
    errors.append(f"❌ Soil profile template not found: {SP_TEMPLATE}")

if errors:
    raise FileNotFoundError("Fix the errors above and re-run this cell")

VALIDATING PATHS
✓ Land use DFS2:  landuse1_clo.dfs2
✓ LU template:    LU_template.csv
✓ Template 1:     Constants_template.csv
✓ Template 2:     InitConditions_template.csv
✓ Output dir:     C:\Users\jaan\Repos\phishes-pdp\plant-growth-module\output_data

✓ ALL PATHS VALID - Ready to process



## 📍 Step 1: Load Land Use Data

**Read spatial land use grid and create code-to-species mapping.**


### Step 1.A: Load Spatial Grids

Read the land use and soil profile DFS2 files and extract 2D grid data. Displays:
- Grid dimensions (rows × columns)
- Unique land use codes present in the land use map
- Unique soil profile codes present in the soil profile map
- Validation that both grids share the same shape


In [11]:
# Read land use DFS2 file
print("Loading land use DFS2 file...")
landuse_ds = mikeio.Dfs2(LANDUSE_DFS2)
landuse_data = landuse_ds.read()[
    0
].to_numpy()  # Get the first (and typically only) item

print(f"\nLand use grid shape:    {landuse_data.shape}")
print(f"Unique land use codes:  {np.unique(landuse_data)}")

# Read soil profile DFS2 file
print("\nLoading soil profile DFS2 file...")
soilprofile_ds = mikeio.Dfs2(SOILPROFILE_DFS2)
soilprofile_data = soilprofile_ds.read()[0].to_numpy()

print(f"\nSoil profile grid shape:   {soilprofile_data.shape}")
print(f"Unique soil profile codes: {np.unique(soilprofile_data)}")

if landuse_data.shape != soilprofile_data.shape:
    raise ValueError("Land use and soil profile grids must have the same shape")

Loading land use DFS2 file...

Land use grid shape:    (1, 140, 125)
Unique land use codes:  [ 1.  2.  3.  4.  5.  7.  9. 10. 12. 16. 17. 18. 19. nan]

Loading soil profile DFS2 file...

Soil profile grid shape:   (1, 140, 125)
Unique soil profile codes: [ 1.  2.  3.  4. nan]


### Step 1.B: Create Classification Mappings

Read the land use and soil profile classification templates and create dictionaries used for spatial mapping.

**Operations:**
- Auto-detects CODE/CLASS columns (case-insensitive)
- Reads optional `Apply` column in land use template
- Builds `code_to_species` and `code_to_soilprofile` dictionaries
- Creates a `zero_fill_classes` list for land use classes with `Apply=0`
- Displays mappings for verification


In [12]:
# Read land use classification template
print("\nLoading land use classification...")
lu_df = pd.read_csv(LU_TEMPLATE)

# Identify relevant columns
code_col = find_col(lu_df, VAL_COLS)
class_col = find_col(lu_df, CLASS_COLS)
apply_col = find_col(lu_df, APPLY_COLS)

if code_col is None or class_col is None:
    raise ValueError(
        "Required columns not found in the land use classification template."
    )

# Confirm column detection with user
confirm_payload = {"Code column": code_col, "Class column": class_col}
if apply_col is not None:
    confirm_payload["Apply column"] = apply_col

confirmed = confirm_columns(
    confirm_payload,
    auto_confirm=AUTO_CONFIRM,
    context="Land use classification",
    multi_files=False,
)
if not confirmed:
    raise RuntimeError("User cancelled land use mapping")

# Create dictionary: CODE -> CLASS (speciesID), and classes forced to zero via Apply=0
code_to_species, zero_fill_classes = split_lu_mapping_by_apply(
    lu_df, code_col, class_col, apply_col
)
zero_fill_values = {class_name: 0.0 for class_name in zero_fill_classes}

print("\nCode to Species mapping:")
for code, species in code_to_species.items():
    print(f"  {int(code):4d} → {species}")

if zero_fill_classes:
    print("\nClasses with Apply=0 (forced to 0 in landuse-based maps):")
    for class_name in sorted(zero_fill_classes):
        print(f"  - {class_name}")

# Read soil profile classification template
print("\nLoading soil profile classification...")
sp_df = pd.read_csv(SP_TEMPLATE)
sp_code_col = find_col(sp_df, VAL_COLS)
sp_class_col = find_col(sp_df, CLASS_COLS)

if sp_code_col is None or sp_class_col is None:
    raise ValueError(
        "Required columns not found in the soil profile classification template."
    )

confirmed = confirm_columns(
    {"Code column": sp_code_col, "Soil profile column": sp_class_col},
    auto_confirm=AUTO_CONFIRM,
    context="Soil profile classification",
    multi_files=False,
)
if not confirmed:
    raise RuntimeError("User cancelled soil profile mapping")

code_to_soilprofile = dict(zip(sp_df[sp_code_col], sp_df[sp_class_col]))

print("\nCode to Soil Profile mapping:")
for code, profile_name in code_to_soilprofile.items():
    print(f"  {int(code):4d} → {profile_name}")


Loading land use classification...

📋 DETECTED COLUMNS in Land use classification:
  Code column              : CODE
  Class column             : CLASS
  Apply column             : Apply

✓ AUTO_CONFIRM enabled - proceeding automatically...

Code to Species mapping:
     2 → Clover-grass mixture (with a predominance of clovers)
     5 → Winter wheat
     1 → Mixture for nectar-producing fallow (eco-payment)
    10 → Winter rapeseed
     3 → Winter rapeseed
    19 → Temporary grassland
     4 → Winter rapeseed
     7 → Corn for silage
     9 → Corn for silage
    12 → Winter rapeseed
    16 → Spring barley

Classes with Apply=0 (forced to 0 in landuse-based maps):
  - Corn for silage
  - Mixture for nectar-producing fallow (eco-payment)
  - Spring barley
  - Temporary grassland
  - Winter rapeseed

Loading soil profile classification...

📋 DETECTED COLUMNS in Soil profile classification:
  Code column              : CODE
  Soil profile column      : CLASS

✓ AUTO_CONFIRM enabled - proc

## 🗺️ Step 2: Generate DFS2 Maps

**Process parameter templates and generate spatially distributed DFS2 files.**

For each template (Constants, InitConditions):
- Reads CSV with species-specific parameter values
- Creates DFS2 file for each parameter by mapping values to land use grid
- Generates maps for all parameters defined in the template


### Step 2.A: Process Templates and Generate Maps

**Main processing loop:**

For each CSV template file:
1. Auto-detects columns: ID, variable/constant, value, optional `type`, optional `template`
2. Prompts for column confirmation (unless `AUTO_CONFIRM=True`)
3. Keeps rows with `type=1` when a `type` column is present
4. Determines mapping scope per variable:
   - `landuse` for species/class-based variables
   - `soilprofile` for soil profile-based variables
5. Generates DFS2 map with correct source grid:
   - Land use grid + `Apply=0` filler classes forced to zero
   - Soil profile grid + soil profile classification mapping

**Output:** DFS2 files named `{parameter_name}.dfs2` in OUTPUT_DIR


In [13]:
# Process all configured templates
total_maps_generated = 0


def _norm(value):
    return str(value).strip().lower()


def _type_is_map(series):
    as_text = series.astype(str).str.strip().str.lower()
    as_num = pd.to_numeric(as_text.str.replace(",", ".", regex=False), errors="coerce")
    return as_num.eq(1) | as_text.isin(["1", "1.0", "true", "yes", "y", "map"])


for template_file in TEMPLATE_FILES:
    print(f"\n{'='*60}")
    print(f"Processing: {template_file.name}")
    print(f"{'='*60}")

    # Read template
    df = pd.read_csv(template_file)

    # Find actual column names in the template
    id_col = find_col(df, ID_COLS)
    value_col = find_col(df, VALUE_COLS)
    key_col = find_col(df, KEY_COLS)
    template_col = find_col(df, TEMPLATE_COLS)
    type_col = find_col(df, TYPE_COLS)

    if not all([id_col, value_col, key_col]):
        print(f"\n⚠ Warning: Missing required columns in {template_file.name}")
        print("  Skipping this template.")
        continue

    # Confirm column detection with user
    confirm_payload = {
        "ID column": id_col,
        "Value column": value_col,
        "Variable column": key_col,
    }
    if template_col is not None:
        confirm_payload["Template column"] = template_col
    if type_col is not None:
        confirm_payload["Type column"] = type_col

    confirmed = confirm_columns(
        confirm_payload,
        auto_confirm=AUTO_CONFIRM,
        context=template_file.name,
    )
    if not confirmed:
        continue

    # Keep only map-generating rows when type column exists (type=1)
    if type_col is not None:
        type_mask = _type_is_map(df[type_col])
        print(f"  Rows with map-enabled type: {int(type_mask.sum())} / {len(df)}")
        df = df[type_mask]

    # Filter only relevant rows
    keep_cols = [id_col, key_col, value_col]
    if template_col is not None:
        keep_cols.append(template_col)

    df = df[keep_cols].dropna(subset=[id_col, key_col, value_col])

    if df.empty:
        print("  No rows to process after filtering. Skipping.")
        continue

    maps_count = 0

    # Group by key name (constant or variable) and generate maps
    for key_name in df[key_col].unique():
        subset = df[df[key_col] == key_name]

        # Determine whether this variable should use landuse or soilprofile mapping
        scope = STATE_VARIABLE_SCOPE.get(str(key_name).strip())
        if template_col is not None:
            scope_values = (
                subset[template_col]
                .dropna()
                .astype(str)
                .str.strip()
                .str.lower()
                .unique()
            )
            if len(scope_values) > 0:
                scope = scope_values[0]

        if scope is None:
            scope = "landuse"

        id_values = dict(zip(subset[id_col], subset[value_col]))
        id_values_norm = {_norm(k): v for k, v in id_values.items()}

        print(f"\n  Generating map: {key_name} (scope={scope})")

        if scope == "landuse":
            species_values = id_values
            for species, value in species_values.items():
                print(f"    {value:<10} ← {species}")

            output_path = OUTPUT_DIR.joinpath(f"{key_name}.dfs2")
            generate_dfs2_map(
                landuse_data,
                landuse_ds,
                code_to_species,
                species_values,
                output_path,
                key_name,
                default_species_values=zero_fill_values,
            )
            maps_count += 1

        elif scope == "soilprofile":
            profile_values = {}
            for code, profile_name in code_to_soilprofile.items():
                candidates = {_norm(profile_name), _norm(code)}
                try:
                    code_float = float(code)
                    candidates.add(_norm(code_float))
                    if code_float.is_integer():
                        candidates.add(_norm(int(code_float)))
                except (TypeError, ValueError):
                    pass

                for candidate in candidates:
                    if candidate in id_values_norm:
                        profile_values[profile_name] = id_values_norm[candidate]
                        break

            for profile_name, value in profile_values.items():
                print(f"    {value:<10} ← {profile_name}")

            output_path = OUTPUT_DIR.joinpath(f"{key_name}.dfs2")
            generate_dfs2_map(
                soilprofile_data,
                soilprofile_ds,
                code_to_soilprofile,
                profile_values,
                output_path,
                key_name,
            )
            maps_count += 1

        else:
            print(f"    ⚠ Unknown scope '{scope}' for {key_name}; skipping")

    print(f"\n✓ Generated {maps_count} maps from {template_file.name}")
    total_maps_generated += maps_count

print(f"\n{'='*60}")
print(f"✓ Total: Generated {total_maps_generated} maps from all templates")


Processing: Constants_template.csv

📋 DETECTED COLUMNS in Constants_template.csv:
  ID column                : speciesID
  Value column             : value
  Variable column          : constant
  Type column              : type

✓ AUTO_CONFIRM enabled - proceeding automatically...
  Rows with map-enabled type: 8 / 27

  Generating map: tmax (scope=landuse)
    22.0       ← Clover-grass mixture (with a predominance of clovers)
    22.0       ← Winter wheat
    16.0       ← Spring barley
    25.0       ← Mixture for nectar-producing fallow (eco-payment)
  📁 Created: tmax.dfs2

  Generating map: topt (scope=landuse)
    18.0       ← Clover-grass mixture (with a predominance of clovers)
    18.0       ← Winter wheat
    14.0       ← Spring barley
    22.0       ← Mixture for nectar-producing fallow (eco-payment)
  📁 Created: topt.dfs2

✓ Generated 2 maps from Constants_template.csv

Processing: InitConditions_template.csv

📋 DETECTED COLUMNS in InitConditions_template.csv:
  ID column    

## ✅ Step 3: Summary and Verification

**List all generated files and verify output quality.**

- Lists all `.dfs2` files in output directory with file sizes
- Reads a sample file to verify dimensions and data range
- Confirms geometry matches input land use grid


In [14]:
# List all generated DFS2 files
print("\n" + "=" * 60)
print("SUMMARY: Generated DFS2 Files")
print("=" * 60)

generated_files = sorted([f for f in os.listdir(OUTPUT_DIR) if f.endswith(".dfs2")])
for i, filename in enumerate(generated_files, 1):
    file_path = OUTPUT_DIR.joinpath(filename)
    file_size = os.path.getsize(file_path) / 1024  # KB
    print(f"{i:2d}. {filename:30s} ({file_size:.1f} KB)")

print(f"\nTotal files generated: {len(generated_files)}")
print(f"Output directory: {OUTPUT_DIR}")


SUMMARY: Generated DFS2 Files
 1. LAI.dfs2                       (69.3 KB)
 2. RD.dfs2                        (69.3 KB)
 3. SOC.dfs2                       (69.3 KB)
 4. tmax.dfs2                      (69.3 KB)
 5. topt.dfs2                      (69.3 KB)

Total files generated: 5
Output directory: C:\Users\jaan\Repos\phishes-pdp\plant-growth-module\output_data


In [15]:
# Verification: Read one generated file to check dimensions
if generated_files:
    sample_file = OUTPUT_DIR.joinpath(generated_files[0])
    sample_ds = mikeio.read(sample_file)

    print("\n" + "=" * 60)
    print("VERIFICATION: Sample Output File")
    print("=" * 60)
    print(f"File: {generated_files[0]}")
    print(f"Shape: {sample_ds[0].shape}")
    print(
        f"Data range: [{sample_ds[0].to_numpy().min():.2f}, {sample_ds[0].to_numpy().max():.2f}]"
    )
    print(f"Geometry matches input: {sample_ds.geometry == landuse_ds.geometry}")
    print("\n✓ Script completed successfully!")


VERIFICATION: Sample Output File
File: LAI.dfs2
Shape: (1, 140, 125)
Data range: [0.00, 0.40]
Geometry matches input: True

✓ Script completed successfully!
